# Tests

Notebook for testing various functions. Below: test of `get_r_beam_px` and beam mask visualization.

In [ ]:
import re
from pathlib import Path

import numpy as np

PROJECT_ROOT = Path("/home/mikl/KurchatovCoop")
REPO_DIR = Path("/home/mikl/KurchatovCoop/repos")

DATA_DIR = PROJECT_ROOT / "data" / "AgBh"
OUT_DIR = PROJECT_ROOT / "debug" / "debug_r_beam_px"
OUT_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR, OUT_DIR

In [ ]:
from autosaxs.processor import get_r_beam_px, calc_beam_abnormal_mask, find_center
from autosaxs.utils import read_from_tiff

# Match *_AgBh<digits>*.*.tif (e.g. 0001_AgBh700_58.3.tif) in data/AgBh
pattern = re.compile(r".*_AgBh\d+.*\.tif$", re.IGNORECASE)
all_tifs = sorted(DATA_DIR.glob("*.tif"))
tif_paths = [p for p in all_tifs if pattern.match(p.name)]
print(f"DATA_DIR = {DATA_DIR}")
print(f"All .tif in dir: {len(all_tifs)} → {[p.name for p in all_tifs[:10]]}{'...' if len(all_tifs) > 10 else ''}")
print(f"Matched *_AgBh<digits>*.tif: {len(tif_paths)} → {[p.name for p in tif_paths]}")

In [ ]:
import os
import matplotlib.pyplot as plt


def view_mask_with_title(img_data, mask, *, tiff_path, r_beam_px, plotFilePath=None):
    """Same 2x2 layout as PLTViewer.view_mask, with a figure title showing r_beam_px."""
    fig, axs = plt.subplots(2, 2, figsize=(32, 24))
    fig.suptitle(f"r_beam_px = {r_beam_px:.1f}", fontsize=14)

    img_data = img_data.astype("float")
    img_data = img_data - min(np.min(img_data), 0)

    axs[0, 0].imshow(np.log1p(img_data), cmap="viridis", origin="lower")
    axs[0, 0].set_title(f"2D SAXS Data: {os.path.basename(tiff_path)}")
    axs[0, 0].set_xlabel("Pixel X")
    axs[0, 0].set_ylabel("Pixel Y")

    axs[0, 1].imshow(mask, cmap="grey", origin="lower")
    axs[0, 1].set_title("Mask")
    axs[0, 1].set_xlabel("Pixel X")
    axs[0, 1].set_ylabel("Pixel Y")

    img_data_masked = np.copy(img_data)
    img_data_masked[mask] = 0.0
    axs[1, 0].imshow(np.log1p(img_data_masked), cmap="viridis", origin="lower")
    axs[1, 0].set_title("Masked 2D SAXS Data, masked as zeros")
    axs[1, 0].set_xlabel("Pixel X")
    axs[1, 0].set_ylabel("Pixel Y")

    img_data_masked[mask] = np.nan
    axs[1, 1].imshow(np.log1p(img_data_masked), cmap="viridis", origin="lower")
    axs[1, 1].set_title("Masked 2D SAXS Data, masked as missing")
    axs[1, 1].set_xlabel("Pixel X")
    axs[1, 1].set_ylabel("Pixel Y")

    if plotFilePath is not None:
        fig.savefig(plotFilePath)
    plt.close(fig)

In [ ]:
from tqdm.auto import tqdm

for tif_path in tqdm(tif_paths):
    image = read_from_tiff(tif_path)
    center_ret = find_center(image)
    center_y_px = center_ret["center_y_px"]
    center_x_px = center_ret["center_x_px"]

    r_beam_px = get_r_beam_px(image, center_y_px, center_x_px)
    if r_beam_px is None:
        r_beam_px = 35.0  # fallback for plotting
    beam_mask = calc_beam_abnormal_mask(
        image, center_y_px, center_x_px, r_beam_px, calc_abnormal_mask=False
    )

    out_path = OUT_DIR / f"{tif_path.stem}.png"
    view_mask_with_title(
        image,
        beam_mask,
        tiff_path=tif_path,
        r_beam_px=r_beam_px,
        plotFilePath=out_path,
    )
    print(f"Saved {out_path}")